# CBMLoss: Concept Bottleneck Models com Mitigação de Concept Leakage
Este notebook está configurado para clonar o repositório no ambiente do Google Colab e persistir todos os **checkpoints, métricas e dados no seu Google Drive** (`/content/drive/MyDrive/CBMLoss_Checkpoints`).

**Como funciona:**
- O repositório é clonado em `/content/CBMLoss`;
- A pasta `checkpoints/` é vinculada diretamente à sua pasta no Google Drive (`CBMLoss_Checkpoints`);
- Você pode colar seus checkpoints antigos manualmente nessa pasta do Drive;
- Todos os novos treinamentos e arquivos gerados (como `leakage_direct_metrics.csv`) ficam salvos permanentemente no seu Google Drive.

In [1]:
# =============================================================================
# 1. MONTAR GOOGLE DRIVE E CONFIGURAR PASTA PERMANENTE DE CHECKPOINTS
# =============================================================================
import os
from google.colab import drive

# 1. Monta o Google Drive
drive.mount('/content/drive')

# Verifica se a GPU está ativa
import torch
print(f'>>> CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'>>> GPU em uso: {torch.cuda.get_device_name(0)}')
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print('>>> ATENÇÃO: GPU não detectada! Ative em: Ambiente de Execução > Alterar tipo de ambiente de execução > T4 GPU')


# 2. Cria a pasta permanente para checkpoints e métricas no Google Drive
GDRIVE_CHECKPOINTS = '/content/drive/MyDrive/CBMLoss_Checkpoints'
os.makedirs(GDRIVE_CHECKPOINTS, exist_ok=True)

print(f'>>> Google Drive montado com sucesso!')
print(f'>>> Pasta persistente de checkpoints: {GDRIVE_CHECKPOINTS}')
print(f'>>> Arquivos atualmente na pasta: {os.listdir(GDRIVE_CHECKPOINTS)}')
print('>>> DICA: Você pode colar os seus checkpoints antigos (.pth) diretamente nessa pasta do Drive!')


Mounted at /content/drive
>>> CUDA disponível: True
>>> GPU em uso: Tesla T4
name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14910 MiB
>>> Google Drive montado com sucesso!
>>> Pasta persistente de checkpoints: /content/drive/MyDrive/CBMLoss_Checkpoints
>>> Arquivos atualmente na pasta: ['checkpoint_latest.pth']
>>> DICA: Você pode colar os seus checkpoints antigos (.pth) diretamente nessa pasta do Drive!


In [2]:
# =============================================================================
# 2. CLONAR OU ATUALIZAR O REPOSITÓRIO GITHUB E VINCULAR CHECKPOINTS
# =============================================================================
import os
import shutil

# 1. Clona ou atualiza o repositório
if not os.path.exists('/content/CBMLoss'):
    !git clone https://github.com/paulohbl/CBMLoss.git /content/CBMLoss
else:
    !cd /content/CBMLoss && git reset --hard && git pull

# 2. Navega para a pasta do projeto
%cd /content/CBMLoss

# 3. Vincula a pasta de checkpoints do repositório ao Google Drive via symlink
GDRIVE_CHECKPOINTS = '/content/drive/MyDrive/CBMLoss_Checkpoints'
if os.path.islink('/content/CBMLoss/checkpoints'):
    os.unlink('/content/CBMLoss/checkpoints')
elif os.path.exists('/content/CBMLoss/checkpoints'):
    for item in os.listdir('/content/CBMLoss/checkpoints'):
        src = os.path.join('/content/CBMLoss/checkpoints', item)
        dst = os.path.join(GDRIVE_CHECKPOINTS, item)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    shutil.rmtree('/content/CBMLoss/checkpoints')

os.symlink(GDRIVE_CHECKPOINTS, '/content/CBMLoss/checkpoints')
print(f'>>> Diretório de trabalho atual: {os.getcwd()}')
print(f'>>> Pasta checkpoints vinculada ao Google Drive: {os.readlink("checkpoints")}')


Cloning into '/content/CBMLoss'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (259/259), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 259 (delta 137), reused 195 (delta 73), pack-reused 0 (from 0)
Receiving objects: 100% (259/259), 3.74 MiB | 25.68 MiB/s, done.
Resolving deltas: 100% (137/137), done.
/content/CBMLoss
>>> Diretório de trabalho atual: /content/CBMLoss
>>> Pasta checkpoints vinculada ao Google Drive: /content/drive/MyDrive/CBMLoss_Checkpoints


In [3]:
# =============================================================================
# 3. INSTALAR DEPENDÊNCIAS DO PROJETO
# =============================================================================
!pip install -r requirements.txt -q
print('>>> Dependências instaladas com sucesso!')


>>> Dependências instaladas com sucesso!


In [4]:
# =============================================================================
# 3.1. CONFIGURAÇÃO DO WEIGHTS & BIASES (WANDB)
# =============================================================================
import os

# OPÇÃO 1: MODO ONLINE (Gráficos ao vivo no wandb.ai)
# Se você ainda não autenticou, pegue sua chave em https://wandb.ai/authorize e descomente a linha abaixo:
# os.environ['WANDB_API_KEY'] = 'COLE_SUA_CHAVE_AQUI'
os.environ['WANDB_MODE'] = 'online'
!wandb online

# OPÇÃO 2: MODO OFFLINE (Sem nuvem, sem login)
# Descomente para rodar offline:
# os.environ['WANDB_MODE'] = 'offline'
# !wandb offline

print(f'>>> WandB configurado para: {os.environ.get("WANDB_MODE")}')


>>> WandB configurado para modo OFFLINE (execução 100% autônoma, sem solicitar input).


In [5]:
# =============================================================================
# 4. DOWNLOAD E PREPARAÇÃO DO DATASET CUB-200-2011
# =============================================================================
# Baixa o CUB-200 (~1.1 GB com barra de progresso) e processa os atributos.
# Se você já colocou 'CUB_200_2011.tgz' no seu Google Drive (CBMLoss_Checkpoints),
# ele copia direto do Drive sem precisar baixar da web!
!python download_datasets.py


1. Generating Synthetic Leaf Dataset on disk...
Generating Synthetic Leaves: 100% 1000/1000 [00:00<00:00, 1234.94it/s]
Synthetic Leaf Dataset generated at data/synthetic_leaf
Baixando CUB-200-2011 (~1.1 GB com barra de progresso)...
CUB_200_2011.tgz: 100% 1.15G/1.15G [00:55<00:00, 20.8MB/s]
Extraindo CUB_200_2011.tgz...
/content/CBMLoss/download_datasets.py:117: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=os.path.dirname(base_dir))
Processing CUB-200-2011 attributes into CSV format...
CUB-200-2011 processing complete! Concept vectors have size 312

All datasets ready for training!


## FASE 2: Estudos de Ablação Desacoplada e Medição Direta de Leakage
Os experimentos abaixo respondem diretamente aos pedidos dos revisores do SIBGRAPI:
1. **Baseline sem Regularização:** $\lambda_{ent}=0.0, \lambda_{ortho}=0.0$
2. **Isolamento da Entropia:** $\lambda_{ent}=0.5, \lambda_{ortho}=0.0$
3. **Isolamento da Descorrelação:** $\lambda_{ent}=0.0, \lambda_{ortho}=0.5$
4. **Configurações Leves Isoladas:** $0.1 / 0.0$ e $0.0 / 0.1$
5. **Medição Direta de Leakage:** Gap contínuo-discreto e Sonda Linear sobre o ruído residual dos conceitos.

*Todos os novos checkpoints e arquivos CSV são salvos automaticamente no seu Google Drive em `CBMLoss_Checkpoints/`.*

In [ ]:
# Baseline: Modelo CBM sem regularização (lambda_ent=0.0, lambda_ortho=0.0)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda


Random seed set to: 42
=== Starting CBMLoss Framework Test on CUB200 Dataset ===
Using device: cuda
Loading dataloaders for cub200...
Dataset specs | Concepts: 312 | Classes: 200
Initializing Model...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 204MB/s]
Setting up Loss -> lambda_ent: 0.0, lambda_ortho: 0.0
wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id py4yviwn.
wandb: Tracking run with wandb version 0.28.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /content/CBMLoss/wandb/offline-run-20260914_213829-py4yviwn
wandb: View this run in the terminal with `wandb leet`
Training Model: cub200_resnet18_ent0.0_ortho0.0_seed42_cub200_resnet18_ent0.0_ortho0.0_seed42...
Epoch 1/100 | LR: 0.000100
Chec

In [ ]:
# Ablação 1: Apenas Entropia (lambda_ent=0.5, lambda_ortho=0.0)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.5 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda


In [ ]:
# Ablação 2: Apenas Descorrelação (lambda_ent=0.0, lambda_ortho=0.5)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.5 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda


In [ ]:
# Ablação 3 (Leve): Apenas Entropia (lambda_ent=0.1, lambda_ortho=0.0)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.1 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda


In [ ]:
# Ablação 4 (Leve): Apenas Descorrelação (lambda_ent=0.0, lambda_ortho=0.1)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.1 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda


### Medição Direta de Concept Leakage
Executa a medição direta (Linear Probing sobre os resíduos $\boldsymbol{r} = \hat{\boldsymbol{c}} - \boldsymbol{c}$ e Discretization Gap $\Delta_{disc} = \text{Acc}_{soft} - \text{Acc}_{hard}$) em todos os modelos salvos em `checkpoints/`. Salva `leakage_direct_metrics.csv` diretamente no seu Google Drive.

In [ ]:
# Medição Direta de Leakage em todos os checkpoints disponíveis no Google Drive
!python measure_leakage.py --dataset cub200 --checkpoint_dir checkpoints --output_csv checkpoints/leakage_direct_metrics.csv --device cuda


## Histórico: Execuções Anteriores (Ablação Conjunta $\lambda_{ent} = \lambda_{ortho}$)
Células abaixo mantidas como referência dos modelos treinados na fase 1.

In [ ]:
# Treino original: lambda=0.0
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda


In [ ]:
# Treino original: lambda=0.1
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.1 --lambda_ortho 0.1 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda


In [ ]:
# Treino original: lambda=0.3
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.3 --lambda_ortho 0.3 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda


In [ ]:
# Treino original: lambda=0.5
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.5 --lambda_ortho 0.5 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda


In [ ]:
# Treino original: lambda=0.7
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.7 --lambda_ortho 0.7 --pretrained --checkpoint_dir checkpoints --patience 5 --device cuda
